# Monolithic Baseline — Bedrock Development Evaluation

This presentation notebook evaluates one tool-using Strands agent with Amazon Nova Lite. AMZN is the development batch and GOOGL is the validation batch. Gold answers are used only after inference.

## Setup

Load the reusable runtime and evaluation code from `src/`.

In [1]:
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found.*")
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

In [2]:
from agent.data import load_question
from agent.monolithic import MonolithicAgentHarness
from agent.observation import format_observable_timeline, format_run_summary
from agent.tools import ProjectTools

In [3]:
from agent.data import load_gold_questions
from pandas import DataFrame
from evaluation.metrics import (
    evaluate_saved_results,
    format_evaluation_summary,
    save_evaluation_results, save_run_records,
)

In [4]:
from config import (
    BEDROCK_NOVA_LITE_MODEL_ID, NOVA_LITE_INPUT_USD_PER_MILLION_TOKENS,
    NOVA_LITE_OUTPUT_USD_PER_MILLION_TOKENS,
)

## Development benchmark

Load the frozen AMZN questions. Gold answers remain separate from agent inputs and appear only in answer comparisons.

In [5]:
ticker = "AMZN"
gold_records = load_gold_questions(ticker)
gold_by_id = {record["question_id"]: record for record in gold_records}

In [6]:
DataFrame([
    {"question_id": record["question_id"], "category": record["category"]}
    for record in gold_records
])

,question_id,category
0,amzn-01,direct_extraction
1,amzn-02,direct_extraction
2,amzn-03,direct_extraction
3,amzn-04,direct_extraction
4,amzn-05,single_formula
5,amzn-06,single_formula
6,amzn-07,single_formula
7,amzn-08,single_formula
8,amzn-09,period_comparison
9,amzn-10,period_comparison


## Preflight

Confirm that Bedrock, Langfuse, and the local Qdrant filing index are available before spending model calls.

In [7]:
harness = MonolithicAgentHarness(
    provider="bedrock", model_id=BEDROCK_NOVA_LITE_MODEL_ID
)
preflight = harness.preflight()

In [8]:
preflight

{'ready': True,
 'provider': 'bedrock',
 'model': 'amazon.nova-lite-v1:0',
 'reasoning_mode': 'none',
 'langfuse': 'authenticated',
 'qdrant_collection': 'sec_filing_chunks_v1'}

## Deterministic tool smoke check

Check fact evidence and Decimal arithmetic before asking the model to use them.

In [9]:
tools = ProjectTools()
facts = tools.lookup_fact(company="AMZN", concept="revenue")

In [10]:
facts

{'status': 'ok',
 'facts': [{'company': 'Amazon.com, Inc.',
   'ticker': 'AMZN',
   'concept': 'revenue',
   'evidence_id': 'fact-amzn-revenue-fy2023',
   'value': '574785000000',
   'unit': 'USD',
   'period': 'FY2023'},
  {'company': 'Amazon.com, Inc.',
   'ticker': 'AMZN',
   'concept': 'revenue',
   'evidence_id': 'fact-amzn-revenue-fy2024',
   'value': '637959000000',
   'unit': 'USD',
   'period': 'FY2024'},
  {'company': 'Amazon.com, Inc.',
   'ticker': 'AMZN',
   'concept': 'revenue',
   'evidence_id': 'fact-amzn-revenue-fy2025',
   'value': '716924000000',
   'unit': 'USD',
   'period': 'FY2025'}]}

In [11]:
calculation = tools.calculate(operation="percentage", inputs=["25", "200"])

In [12]:
calculation

{'status': 'ok', 'result': '12.5'}

## Question 1: direct extraction

Run one question, then inspect its complete audit trail before continuing.

In [13]:
question_1 = load_question("amzn-01")
gold_1 = gold_by_id[question_1.question_id]["answer"]
run_1 = harness.run_question(question_1.question)

In [14]:
run_1.answer

MonolithicAnswer(answer_text="Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.", values=[AnswerValue(label='consolidated net sales', value='716924000000', unit='USD', period='FY2025', source_evidence_ids=['fact-amzn-revenue-fy2025'])], evidence_ids=['fact-amzn-revenue-fy2025'], calculations=[], conclusions=["Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000."])

### Full trace: Question 1

In [15]:
print(format_observable_timeline(run_1.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 1

In [16]:
print(format_run_summary(run_1, gold_1))

Question: What was Amazon.com, Inc.'s consolidated net sales in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.
Gold answer: 716924000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-revenue-fy2025
Latency: 5896.53 ms | Tokens: {'inputTokens': 6385, 'outputTokens': 499, 'totalTokens': 6884} | Retries: 0
Structured-output repairs: 1
Cost: {'input': '0.0003831', 'output': '0.00011976', 'total': '0.00050286'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/f11900b8613feaa5e1ac852a5175871c


### Cost breakdown: Question 1

Nova Lite on-demand text pricing in `us-east-1`, frozen on 2026-08-24.

In [17]:
input_cost = run_1.token_usage["inputTokens"] * NOVA_LITE_INPUT_USD_PER_MILLION_TOKENS / 1_000_000
output_cost = run_1.token_usage["outputTokens"] * NOVA_LITE_OUTPUT_USD_PER_MILLION_TOKENS / 1_000_000
input_cost + output_cost

Decimal('0.00050286')

## Question 2: single formula

Run one formula question, then inspect it before moving on.

In [18]:
question_2 = load_question("amzn-05")
gold_2 = gold_by_id[question_2.question_id]["answer"]
run_2 = harness.run_question(question_2.question)

### Full trace: Question 2

In [19]:
print(format_observable_timeline(run_2.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 2

In [20]:
print(format_run_summary(run_2, gold_2))

Question: What was Amazon.com, Inc.'s operating margin in fiscal 2025?
Model answer: Amazon.com, Inc.'s operating margin in fiscal 2025 was 11.16 percent.
Gold answer: 11.16
Gold values: -
Tools: lookup_fact (ok), lookup_fact (ok), calculate (ok)
Evidence: fact-amzn-operating-income-fy2025, fact-amzn-revenue-fy2025
Latency: 3814.25 ms | Tokens: {'inputTokens': 6805, 'outputTokens': 413, 'totalTokens': 7218} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.0004083', 'output': '0.00009912', 'total': '0.00050742'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/fd92272effe7219390e92b6274ac67f0


## Question 3: period comparison

This example combines filing search, normalized facts, and calculations.

In [21]:
question_3 = load_question("amzn-12")
gold_3 = gold_by_id[question_3.question_id]["answer"]
run_3 = harness.run_question(question_3.question)

### Full trace: Question 3

In [22]:
print(format_observable_timeline(run_3.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 3

In [23]:
print(format_run_summary(run_3, gold_3))

Question: From fiscal 2024 to fiscal 2025, how did AWS operating income growth compare with Amazon's consolidated operating income growth?
Model answer: From fiscal 2024 to fiscal 2025, AWS operating income grew by 118.63% while Amazon's consolidated operating income grew by 116.59%.
Gold answer: consolidated operating income grew faster than AWS operating income; the growth-rate difference was 2.10 percentage points
Gold values: AWS operating income growth: 14.49 percent, consolidated operating income growth: 16.59 percent
Tools: search_filing (ok), search_filing (ok), calculate (ok), calculate (ok)
Evidence: filing-amzn-2025-page-0068-table-01
Latency: 8803.83 ms | Tokens: {'inputTokens': 27628, 'outputTokens': 635, 'totalTokens': 28263} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.00165768', 'output': '0.0001524', 'total': '0.00181008'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/2d823b2acdbce0ee3b7de9757f82d5db


## Question 4: multi-step calculation

The final dry run checks a longer chain before the complete batch.

In [24]:
question_4 = load_question("amzn-16")
gold_4 = gold_by_id[question_4.question_id]["answer"]
run_4 = harness.run_question(question_4.question)

### Full trace: Question 4

In [25]:
print(format_observable_timeline(run_4.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its requested period and
  unit; before calculating, confirm every input has the requested period. Do not
  guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth, calculate (new - old) / old
  * 100, not new / old * 100. Keep every used calculator operation, exact inputs,
  and exact result in calculations. Ensure directional wording agrees with results.
- Ground each reported value in returned evidence or a calculator result. Cite only
  returned evidence IDs that directly suppor

### Run summary: Question 4

In [26]:
print(format_run_summary(run_4, gold_4))

Question: Did Amazon's net sales, operating income, operating cash flow, and total assets grow at similar rates from fiscal 2024 to fiscal 2025?
Model answer: Amazon's net sales, operating income, operating cash flow, and total assets did not grow at similar rates from fiscal 2024 to fiscal 2025.
Gold answer: all four measures increased; total assets grew fastest; net sales grew slowest; the spread between the fastest and slowest growth rates was 18.53 percentage points
Gold values: net sales growth: 12.38 percent, operating income growth: 16.59 percent, operating cash flow growth: 20.40 percent, total assets growth: 30.91 percent
Tools: lookup_fact (ok), lookup_fact (ok), lookup_fact (ok), lookup_fact (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok), calculate (ok)
Evidence: fact-amzn-revenue-fy2024, fact-amzn-revenue-fy2025, fact-amzn-operating-income-fy2024, fact-amzn-operating-income-fy2025, fact-amzn-operating-cash-flow-fy2024, fact-amzn-operati

## Full AMZN development batch

Run all 16 development questions with Nova Lite. Each attempt is appended to the official local JSONL run log.

In [27]:
batch_runs = harness.run_batch(ticker)
assert [run.question.question_id for run in batch_runs] == [gold["question_id"] for gold in gold_records]
results_dir = save_run_records(batch_runs)

In [28]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status}
    for run in batch_runs
])

,question_id,status
0,amzn-01,success
1,amzn-02,success
2,amzn-03,success
3,amzn-04,success
4,amzn-05,success
5,amzn-06,success
6,amzn-07,success
7,amzn-08,success
8,amzn-09,success
9,amzn-10,success


## Inspect every answer

The compact view makes incorrect values, missing evidence, failed calls, and unexpectedly expensive questions visible before looking at aggregate scores.

In [29]:
for run in batch_runs:
    gold = gold_by_id[run.question.question_id]["answer"]
    print(format_run_summary(run, gold))
    print("-" * 88)

Question: What was Amazon.com, Inc.'s consolidated net sales in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated net sales in fiscal 2025 was $716,924,000,000.
Gold answer: 716924000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-revenue-fy2025
Latency: 2749.09 ms | Tokens: {'inputTokens': 3971, 'outputTokens': 238, 'totalTokens': 4209} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.00023826', 'output': '0.00005712', 'total': '0.00029538'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/2ea26114406a173cdf2105df20e06f61
----------------------------------------------------------------------------------------
Question: What was Amazon.com, Inc.'s consolidated operating income in fiscal 2025?
Model answer: Amazon.com, Inc.'s consolidated operating income in fiscal 2025 was $79,975,000,000.
Gold answer: 79975000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-amzn-operating-income-fy2025
Latency: 2308.71 ms | Tokens:

## Deterministic evaluation

Gold values, calculation steps, required tools, and saved tool events are compared with ordinary Python. Structured final values must match the gold number and unit exactly, and extra final values fail. Formatting is scored separately across the structured values and answer text.

Retrieval is measured from saved tool events. Haiku 4.5 optionally adds qualitative 1–5 workflow scores without overriding deterministic numerical correctness.

In [30]:
saved_evaluation = evaluate_saved_results(
    latest=True, architecture=batch_runs[0].architecture, ticker=ticker,
)
assert saved_evaluation["results_directory"] == results_dir
question_metrics = saved_evaluation["question_rows"]
retrieval_metrics = saved_evaluation["retrieval_rows"]
summary = saved_evaluation["summary"]

## Exploratory Haiku 4.5 workflow review

These qualitative scores support diagnosis only. They cannot override deterministic numerical or audit failures.

In [31]:
saved_evaluation = evaluate_saved_results(
    latest=True, rerun_judge=True,
    architecture=batch_runs[0].architecture, ticker=ticker,
)
assert saved_evaluation["results_directory"] == results_dir
judge_metrics = saved_evaluation["judge_rows"] or []
summary = saved_evaluation["summary"]

In [32]:
DataFrame(judge_metrics, columns=(
    "question_id", "llm_answer_correctness_grade", "workflow_correctness",
    "chronological_coherence", "workflow_efficiency", "auditability",
))

,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability
0,amzn-01,5,5,5,5,5
1,amzn-02,5,5,5,5,5
2,amzn-03,4,4,5,3,3
3,amzn-04,2,2,5,5,4
4,amzn-05,5,5,5,5,5
5,amzn-06,5,5,5,5,5
6,amzn-07,5,5,5,5,5
7,amzn-08,5,5,5,5,5
8,amzn-09,1,1,2,3,4
9,amzn-10,1,1,4,3,4


### AMZN questions needing review

Only runs with a failed deterministic check appear here.

In [33]:
CHECKS = ("numerical_correct", "evidence_valid", "evidence_complete",
          "value_evidence_valid", "periods_valid", "formatting_correct",
          "audit_trail_complete")
failed_question_metrics = [row for row in question_metrics if row["status"] != "success"
                           or any(row[name] is not True for name in CHECKS)]
DataFrame(failed_question_metrics, columns=(
    "question_id", "category", "numerical_correct",
    "evidence_complete", "formula_correct", "formatting_correct",
    "audit_trail_complete",
))

,question_id,category,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,amzn-03,direct_extraction,False,True,None,False,False
1,amzn-04,direct_extraction,False,True,None,False,False
2,amzn-09,period_comparison,False,True,False,True,False
3,amzn-10,period_comparison,False,True,False,True,False
4,amzn-11,period_comparison,False,True,False,True,False
5,amzn-12,period_comparison,False,True,False,False,False
6,amzn-13,multi_step,False,True,False,False,False
7,amzn-14,multi_step,False,True,False,True,False
8,amzn-15,multi_step,False,True,False,True,False
9,amzn-16,multi_step,False,True,False,False,False


### Metrics by question type

Category scores show whether errors are concentrated in extraction, one formula, comparisons, or longer calculation chains.

In [34]:
DataFrame(summary["by_category"]).T.rename_axis("question_type")

,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,,,,,
direct_extraction,4.0,0.5,1.0,NaN,0.50
multi_step,4.0,0.0,1.0,0.0,0.50
period_comparison,4.0,0.0,1.0,0.0,0.75
single_formula,4.0,1.0,1.0,1.0,1.00


### Final metrics

Higher is better except for unsupported-number rate and irrelevant retrieval rate. Evidence validity checks whether cited IDs were returned; evidence completeness checks whether every final value cites returned source evidence. Retrieval scores each search call's top five independently.

In [35]:
print(format_evaluation_summary(summary))

Runs: 16
run_success_rate: 100.00%
numerical_correctness_rate: 37.50%
evidence_validity_rate: 100.00%
evidence_completeness_rate: 100.00%
formula_correctness_rate: 33.33%
formatting_correctness_rate: 68.75%
unsupported_number_rate: 37.50%
audit_trail_completeness_rate: 37.50%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 96.55%
tool_recall: 93.33%
tool_f1: 94.92%
tool_exact_match_rate: 81.25%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 96.83%
conditional_execution_rate: 93.75%
evidence_recall_at_5: 100.00%
irrelevant_retrieval_rate_at_5: 57.50%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 12.50%
Average latency: 6008.98 ms
Input tokens: 173182
Output tokens: 11412
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 2
Infrastructure failures: 0
Estimated cost: $0.01312980
Judge llm answer correctness grade: 2.94/5
Judge workflow correctness: 3.06/5
Judge chronological coherence: 4.0

### AMZN findings and conclusion

In this Nova Lite batch, execution and grounding were reliable: all 16 runs completed, evidence validity, evidence completeness, and Evidence Recall@5 were 100%, and Tool F1 was 94.92%. Retrieval still contained noise (57.50% irrelevant-retrieval rate), but the required evidence was consistently available.

Numerical correctness was 37.50%, while formula and audit-trail correctness were 33.33% and 37.50%. The model answered the normalized direct-fact questions and all four single-formula questions correctly. The review table contains ten questions; all had complete evidence, so their failures came after retrieval.

Every AMZN review row explained:

- **AMZN-03 — fails numerical, formatting, and audit:** the cited filing tables report values in millions, but the structured answer stored `45606 USD` and the text said `$45,606 million`. The required absolute value is `45606000000 USD`.
- **AMZN-04 — fails numerical, formatting, and audit:** the same scaling error produced `4750 USD`/`$4,750 million` instead of the required `4750000000 USD`.
- **AMZN-09 — fails numerical, formula, and audit:** `percentage(old, new)` returned old/new ratios rather than growth. The answer gave 88.99% and 15.77%; gold requires 12.38% net-sales growth and 16.59% operating-income growth.
- **AMZN-10 — fails numerical, formula, and audit:** the model again calculated old/new ratios, reporting 76.28% and 83.06% instead of growth rates of 31.09% and 20.40%; it therefore also chose the wrong faster-growing measure.
- **AMZN-11 — fails numerical, formula, and audit:** the absolute change used FY2024 minus FY2023, returning `$97,040,000,000` instead of FY2025 minus FY2024, `$193,148,000,000`. Its final 30.91% was not reproduced by the recorded calculator event, which returned the new/old ratio 130.91%.
- **AMZN-12 — fails numerical, formula, formatting, and audit:** the model used the wrong AWS periods and treated new/old ratios as growth. The structured AWS value was 18.63% while the prose said 11.9%; gold requires 14.49%. The prose and structured values also disagreed.
- **AMZN-13 — fails numerical, formula, formatting, and audit:** the question requires four annual margins—10.75%, 11.16%, 9.29%, and 10.83%—but the answer returned only decimal changes, `0.0040%` and `0.0155%`, and did not display the required values.
- **AMZN-14 — fails numerical, formula, and audit:** the answer returned only a 3.39% change and said the share increased. Gold requires the two annual shares, 58.07% and 57.03%, which show a decline.
- **AMZN-15 — fails numerical, formula, and audit:** after summing the three cash-flow values correctly, the model divided FY2025 cash flow alone by three. It returned only 178.48%; gold requires the `$113,445,666,666.67` average, `$26,068,333,333.33` difference, and 22.98%.
- **AMZN-16 — fails numerical, formula, formatting, and audit:** the response mixed old/new ratios with growth and missed strict two-decimal targets: 88.99%, 16.59%, 20.41%, and 30.92% were returned instead of 12.38%, 16.59%, 20.40%, and 30.91%. The prose stated only a conclusion and did not display the structured values.

The main AMZN bottleneck was therefore financial calculation and unit handling, not access to evidence. The separate judge cost was $0.2012, compared with $0.0131 for the Nova answer batch.

## Save evaluation files

The run records were saved before evaluation. Add the computed metric and judge files to that same result folder.

In [36]:
results_dir = save_evaluation_results(
    results_dir, question_metrics, summary, retrieval_metrics,
    judge_rows=judge_metrics or None,
)
results_dir

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-08-31T151912559431Z-monolithic-amazon-nova-lite-v1-0-AMZN')

In [37]:
DataFrame({"saved_file": sorted(path.name for path in results_dir.iterdir())})

,saved_file
0,judge_records.jsonl
1,question_metrics.csv
2,retrieval_metrics.csv
3,runs.jsonl
4,summary.json


## Failures

Provider and tool failures remain experimental results. They are displayed rather than silently rerun or repaired.

In [38]:
DataFrame(
    [
        {"question_id": run.question.question_id,
         "failure_category": run.failure_category,
         "failure_message": run.failure_message}
        for run in batch_runs if run.status == "failure"
    ],
    columns=("question_id", "failure_category", "failure_message"),
)

,question_id,failure_category,failure_message


# Validation batch: Alphabet (GOOGL)

Run the frozen validation company after reviewing the AMZN development results.

In [39]:
validation_ticker = "GOOGL"
validation_gold_records = load_gold_questions(validation_ticker)
validation_gold_by_id = {
    record["question_id"]: record for record in validation_gold_records
}

In [40]:
validation_runs = harness.run_batch(validation_ticker)
assert len(validation_runs) == len(validation_gold_records)
validation_results_dir = save_run_records(validation_runs)

In [41]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status}
    for run in validation_runs
])

,question_id,status
0,googl-01,success
1,googl-02,success
2,googl-03,success
3,googl-04,success
4,googl-05,success
5,googl-06,success
6,googl-07,success
7,googl-08,success
8,googl-09,success
9,googl-10,success


## Inspect every answer

Use the same compact answer and gold comparison as the AMZN batch.

In [42]:
for run in validation_runs:
    gold = validation_gold_by_id[run.question.question_id]["answer"]
    print(format_run_summary(run, gold))
    print("-" * 88)

Question: What were Alphabet Inc.'s consolidated revenues in fiscal 2025?
Model answer: Alphabet Inc.'s consolidated revenues in fiscal 2025 were $402,836 million.
Gold answer: 402836000000
Gold values: -
Tools: search_filing (ok)
Evidence: filing-googl-2025-page-0033-table-02
Latency: 3668.83 ms | Tokens: {'inputTokens': 7706, 'outputTokens': 337, 'totalTokens': 8043} | Retries: 0
Structured-output repairs: 0
Cost: {'input': '0.00046236', 'output': '0.00008088', 'total': '0.00054324'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/f5cfc74df3745a7e81adbe33a9dc837b
----------------------------------------------------------------------------------------
Question: What was Alphabet Inc.'s consolidated income from operations in fiscal 2025?
Model answer: Alphabet Inc.'s consolidated income from operations in fiscal 2025 was $129,039,000,000.
Gold answer: 129039000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-googl-operating-income-fy2025
Latency: 236

## Deterministic evaluation

In [43]:
validation_saved_evaluation = evaluate_saved_results(
    latest=True, architecture=validation_runs[0].architecture,
    ticker=validation_ticker,
)
assert validation_saved_evaluation["results_directory"] == validation_results_dir
validation_question_metrics = validation_saved_evaluation["question_rows"]
validation_retrieval_metrics = validation_saved_evaluation["retrieval_rows"]
validation_summary = validation_saved_evaluation["summary"]

## Exploratory Haiku 4.5 workflow review

These qualitative scores support diagnosis only. They cannot override deterministic numerical or audit failures.

In [44]:
validation_saved_evaluation = evaluate_saved_results(
    latest=True, rerun_judge=True,
    architecture=validation_runs[0].architecture, ticker=validation_ticker,
)
assert validation_saved_evaluation["results_directory"] == validation_results_dir
validation_judge_metrics = validation_saved_evaluation["judge_rows"] or []
validation_summary = validation_saved_evaluation["summary"]

In [45]:
DataFrame(validation_judge_metrics, columns=(
    "question_id", "llm_answer_correctness_grade", "workflow_correctness",
    "chronological_coherence", "workflow_efficiency", "auditability",
))

,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability
0,googl-01,2,2,5,5,3
1,googl-02,5,5,5,4,5
2,googl-03,3,2,5,2,2
3,googl-04,3,3,5,5,3
4,googl-05,5,5,5,5,5
5,googl-06,5,5,5,5,4
6,googl-07,5,5,5,5,5
7,googl-08,5,5,5,5,5
8,googl-09,1,1,2,3,4
9,googl-10,1,1,5,4,5


### GOOGL questions needing review

Only GOOGL runs with a failed deterministic check appear in this table.

In [46]:
validation_failed_question_metrics = [
    row for row in validation_question_metrics if row["status"] != "success"
    or any(row[name] is not True for name in CHECKS)
]
DataFrame(validation_failed_question_metrics, columns=(
    "question_id", "category", "numerical_correct",
    "evidence_complete", "formula_correct", "formatting_correct",
    "audit_trail_complete",
))

,question_id,category,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,googl-01,direct_extraction,False,True,None,False,False
1,googl-03,direct_extraction,False,True,None,False,False
2,googl-04,direct_extraction,False,True,None,False,False
3,googl-06,single_formula,True,True,False,False,False
4,googl-09,period_comparison,False,True,False,False,False
5,googl-10,period_comparison,False,True,False,True,False
6,googl-11,period_comparison,True,True,False,True,False
7,googl-12,period_comparison,False,True,False,True,False
8,googl-13,multi_step,False,True,False,False,False
9,googl-14,multi_step,False,True,False,True,False


### Metrics by question type

In [47]:
DataFrame(validation_summary["by_category"]).T.rename_axis("question_type")

,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,,,,,
direct_extraction,4.0,0.25,1.0,NaN,0.25
multi_step,4.0,0.00,1.0,0.00,0.50
period_comparison,4.0,0.25,1.0,0.00,0.75
single_formula,4.0,1.00,1.0,0.75,0.75


### Final metrics

In [48]:
print(format_evaluation_summary(validation_summary))

Runs: 16
run_success_rate: 100.00%
numerical_correctness_rate: 37.50%
evidence_validity_rate: 100.00%
evidence_completeness_rate: 100.00%
formula_correctness_rate: 25.00%
formatting_correctness_rate: 56.25%
unsupported_number_rate: 20.83%
audit_trail_completeness_rate: 25.00%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 89.66%
tool_recall: 86.67%
tool_f1: 88.14%
tool_exact_match_rate: 68.75%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 96.88%
conditional_execution_rate: 85.42%
evidence_recall_at_5: 100.00%
irrelevant_retrieval_rate_at_5: 32.50%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 12.50%
Average latency: 6644.92 ms
Input tokens: 213683
Output tokens: 12303
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 2
Infrastructure failures: 0
Estimated cost: $0.01577370
Judge llm answer correctness grade: 2.75/5
Judge workflow correctness: 2.69/5
Judge chronological coherence: 4.3

### GOOGL findings and conclusion

In this Nova Lite validation batch, all 16 runs completed, evidence validity, evidence completeness, and Evidence Recall@5 were 100%, and Tool F1 was 88.14%. Retrieval was less noisy than for AMZN, with a 32.50% irrelevant-retrieval rate, although tool selection and execution were slightly weaker.

Numerical correctness was 37.50%, and formula and audit-trail correctness were both 25.00%. The review table contains twelve questions. Evidence completeness passed for every row; two rows have numerically correct final answers but remain here because their formula or audit trail failed.

Every GOOGL review row explained:

- **GOOGL-01 — fails numerical, formatting, and audit:** a filing value expressed in millions was stored as `402836 USD` and displayed as `$402,836 million`; gold requires `402836000000 USD`.
- **GOOGL-03 — fails numerical, formatting, and audit:** the correct Google Services tables were cited, but `139404 USD` was stored instead of `139404000000 USD`. Several unnecessary subtraction calls did not fix the scale.
- **GOOGL-04 — fails numerical, formatting, and audit:** `13910 USD`/`$13,910 million` was returned instead of the absolute value `13910000000 USD`.
- **GOOGL-06 — numerical passes; formula, formatting, and audit fail:** the final 32.81% margin is correct, but the trace contains only `divide(132170, 402836) = 0.328...`, not the expected percentage operation. The prose also introduces `$132,170 million` and `$402,836 million`, violating the absolute-number display rule.
- **GOOGL-09 — fails numerical, formula, formatting, and audit:** 32.11% is an operating margin, not revenue growth, and 114.81% is a new/old ratio. Gold requires 15.09% revenue growth and 14.81% operating-income growth; the prose omits those numbers.
- **GOOGL-10 — fails numerical, formula, and audit:** old/new ratios of 75.75% and 76.07% were reported instead of net-income and cash-flow growth of 32.01% and 31.46%, reversing the comparison.
- **GOOGL-11 — numerical and formatting pass; formula and audit fail:** the final `$145,025,000,000` and 32.21% are correct, but the calculator returned 132.21% and no recorded subtraction converted that ratio to growth. The correct final percentage is therefore unsupported by the trace.
- **GOOGL-12 — fails numerical, formula, and audit:** the model returned old/new ratios of 86.99% and 87.10% instead of growth rates of 14.96% and 14.81%.
- **GOOGL-13 — fails numerical, formula, formatting, and audit:** it returned only decimal changes, `0.0033%` and `-0.0421%`, rather than the four required annual margins. The signs and percent scaling were also wrong, and the required values were absent from the prose.
- **GOOGL-14 — fails numerical, formula, and audit:** the answer supplied only an 86.99% old/new ratio. Gold requires annual Google Services shares of 107.89% and 108.03%; neither was returned.
- **GOOGL-15 — fails numerical, formula, and audit:** the correct three-year sum was `391758000000`, but the next calculation divided an unrelated `149616000000` by three. It returned only 230.27%; gold requires a `$130,586,000,000.00` average, `$34,127,000,000.00` difference, and 26.13%.
- **GOOGL-16 — fails numerical, formula, formatting, and audit:** all four outputs were old/new ratios—86.89%, 87.10%, 81.20%, and 75.64%—rather than growth rates of 15.09%, 14.81%, 31.46%, and 32.21%. The prose again omitted the structured values.

The judge again rated auditability (3.69/5) above answer correctness (2.75/5). The traces were inspectable, but Nova Lite was not consistently precise enough for period comparisons and multi-step financial reasoning. Judge cost was $0.2061 versus $0.0158 for the Nova answer batch.

## Save evaluation files

In [49]:
validation_results_dir = save_evaluation_results(
    validation_results_dir, validation_question_metrics, validation_summary,
    validation_retrieval_metrics, judge_rows=validation_judge_metrics,
)
validation_results_dir

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-08-31T152441084692Z-monolithic-amazon-nova-lite-v1-0-GOOGL')

In [50]:
DataFrame({"saved_file": sorted(path.name for path in validation_results_dir.iterdir())})

,saved_file
0,judge_records.jsonl
1,question_metrics.csv
2,retrieval_metrics.csv
3,runs.jsonl
4,summary.json


## Failures

In [51]:
DataFrame(
    [
        {"question_id": run.question.question_id,
         "failure_category": run.failure_category,
         "failure_message": run.failure_message}
        for run in validation_runs if run.status == "failure"
    ],
    columns=("question_id", "failure_category", "failure_message"),
)

,question_id,failure_category,failure_message


## Overall Nova Lite conclusion

Across AMZN and GOOGL, the monolithic Nova Lite agent's main weakness was not evidence access: every run completed, evidence completeness and Evidence Recall@5 were 100%, and the saved traces usually showed the correct source facts. The dominant problem was applying financial meaning after retrieval.

The most frequent errors were failing to convert filing values reported in millions into absolute USD, treating old/new or new/old ratios as growth rates, selecting the wrong fiscal-period operands, and returning only one comparison value when the question required several intermediate and final values. Some answers were numerically correct but still failed formula or audit checks because the recorded calculator calls did not reproduce the final number. Formatting failures mostly came from magnitude words such as `million` or from prose that omitted or contradicted the structured values.

Performance therefore declined as reasoning depth increased: normalized direct facts and single-formula questions were strongest, while period comparisons and multi-step questions produced most errors. For this model, the primary reliability risk is calculation semantics and multi-step completeness, followed by unit scaling—not hallucinated evidence.